[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_03/MFM_ch3_two_pass_inference/MFM_ch3_two_pass_inference.ipynb)

# MFM_ch3_two_pass_inference

Lecture 3 and Seminar 3 (B9-B11): GRS as a Sharpe-ratio test, misspecification-robust two-pass standard errors (Kan-Robotti-Shanken), a useless-factor simulation (Kan-Zhang), cross-sectional OLS and GLS R2 with industry portfolios (Lewellen-Nagel-Shanken), the number of factors (Kaiser, Bai-Ng, Ahn-Horenstein), model comparison by maximum Sharpe ratios with a stationary bootstrap (Barillas-Shanken), and GRS with bootstrap and GMM Wald inference.

*Modelling Financial Markets (MFM), Chapter 3: Factor Models and Asset Pricing. Author: Daniel Traian Pele.*

In [ ]:
import os
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Data

Market prices from the course repository (`data/market`); factor and portfolio returns from the Kenneth French Data Library.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

SECTORS = ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']        # from Dec 1998
SECTORS_ALL = SECTORS + ['XLRE', 'XLC']                                           # XLRE 2015, XLC 2018
SECTOR_NAMES = {'XLB': 'Materials', 'XLE': 'Energy', 'XLF': 'Financials', 'XLI': 'Industrials',
                'XLK': 'Technology', 'XLP': 'Cons. staples', 'XLU': 'Utilities', 'XLV': 'Health care',
                'XLY': 'Cons. discretionary', 'XLRE': 'Real estate', 'XLC': 'Communication'}
FACTOR_ETFS = ['MTUM', 'VLUE', 'QUAL', 'USMV', 'IWM', 'IWD', 'IWF']
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'FP', 'SNG', 'H2O', 'SNN', 'EL', 'DIGI', 'TEL']
BVB_NAMES = {'TLV': 'Banca Transilvania', 'SNP': 'OMV Petrom', 'BRD': 'BRD-GSG', 'TGN': 'Transgaz',
             'FP': 'Fondul Proprietatea', 'SNG': 'Romgaz', 'H2O': 'Hidroelectrica', 'SNN': 'Nuclearelectrica',
             'EL': 'Electrica', 'DIGI': 'Digi', 'TEL': 'Transelectrica'}

_CACHE = {}


def read_market(symbol):
    """Daily prices of one asset from the course data."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(symbol):
    """Price used: adjusted close for ETFs and stocks, close for indices."""
    d = read_market(symbol)
    col = 'close' if symbol.endswith('.INDX') or symbol == 'BET' else 'adjusted_close'
    s = d[col].astype(float)
    return s[s > 0].rename(symbol)


def prices(symbols, start=None, end=None):
    """Prices aligned on the common days (multi-asset analyses align prices before taking returns)."""
    p = pd.concat([price(s) for s in symbols], axis=1).dropna()
    return p.loc[start:end]


def log_returns(p):
    """Log returns of an already aligned table (common days)."""
    return np.log(p).diff().dropna()


# =============================================================================
# KENNETH FRENCH DATA LIBRARY
# =============================================================================
FILES = {
    ('ff3', 'M'): 'F-F_Research_Data_Factors_CSV.zip',
    ('ff3', 'D'): 'F-F_Research_Data_Factors_daily_CSV.zip',
    ('ff5', 'M'): 'F-F_Research_Data_5_Factors_2x3_CSV.zip',
    ('ff5', 'D'): 'F-F_Research_Data_5_Factors_2x3_daily_CSV.zip',
    ('mom', 'M'): 'F-F_Momentum_Factor_CSV.zip',
    ('mom', 'D'): 'F-F_Momentum_Factor_daily_CSV.zip',
    ('p25', 'M'): '25_Portfolios_5x5_CSV.zip',
    ('ind10', 'M'): '10_Industry_Portfolios_CSV.zip',
    ('ind30', 'M'): '30_Industry_Portfolios_CSV.zip',
    ('p100', 'M'): '100_Portfolios_10x10_CSV.zip',
    # univariate sorts (first table: value-weighted), Jensen-Kelly-Pedersen case study
    ('ME', 'M'): 'Portfolios_Formed_on_ME_CSV.zip',
    ('BE-ME', 'M'): 'Portfolios_Formed_on_BE-ME_CSV.zip',
    ('OP', 'M'): 'Portfolios_Formed_on_OP_CSV.zip',
    ('INV', 'M'): 'Portfolios_Formed_on_INV_CSV.zip',
    ('E-P', 'M'): 'Portfolios_Formed_on_E-P_CSV.zip',
    ('CF-P', 'M'): 'Portfolios_Formed_on_CF-P_CSV.zip',
    ('D-P', 'M'): 'Portfolios_Formed_on_D-P_CSV.zip',
    ('AC', 'M'): 'Portfolios_Formed_on_AC_CSV.zip',
    ('NI', 'M'): 'Portfolios_Formed_on_NI_CSV.zip',
    ('BETA', 'M'): 'Portfolios_Formed_on_BETA_CSV.zip',
    ('VAR', 'M'): 'Portfolios_Formed_on_VAR_CSV.zip',
    ('RESVAR', 'M'): 'Portfolios_Formed_on_RESVAR_CSV.zip',
    ('PRIOR_12_2', 'M'): '10_Portfolios_Prior_12_2_CSV.zip',
    ('PRIOR_1_0', 'M'): '10_Portfolios_Prior_1_0_CSV.zip',
    ('PRIOR_60_13', 'M'): '10_Portfolios_Prior_60_13_CSV.zip',
}


def _parse_first_table(text, date_len):
    """First table of a French data set: header ',col1,col2...', then rows YYYYMM(DD),values."""
    lines = text.splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if not parts or len(parts[0]) != date_len or not parts[0].isdigit():
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    fmt = '%Y%m' if date_len == 6 else '%Y%m%d'
    df['date'] = pd.to_datetime(df['date'], format=fmt)
    if date_len == 6:
        df['date'] = df['date'] + pd.offsets.MonthEnd(0)
    df = df.set_index('date')
    return df.replace([-99.99, -999.0], np.nan) / 100.0          # percent -> decimal


def french(name='ff5', freq='M'):
    """Fama-French factors / momentum / portfolios, in decimals (0.01 = 1%)."""
    key = (name, freq)
    if key in _CACHE:
        return _CACHE[key]
    url = FRENCH + FILES[key]
    raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode('latin-1')
    df = _parse_first_table(text, 6 if freq == 'M' else 8)
    df.columns = [c.replace('Mom', 'MOM').strip() for c in df.columns]
    _CACHE[key] = df
    return df


def factors(freq='M'):
    """Mkt-RF, SMB, HML, RMW, CMA, RF, MOM and SMB_FF3 over the common period.

    The five-factor SMB (sorts on B/M, profitability and investment) enters FF5;
    SMB_FF3 is the published three-factor SMB (2x3 sorts on size and B/M),
    used in FF3 and Carhart. HML, Mkt-RF and RF are the same in the two data sets.
    """
    f3 = french('ff3', freq)[['SMB']].rename(columns={'SMB': 'SMB_FF3'})
    return pd.concat([french('ff5', freq), french('mom', freq), f3], axis=1).dropna()


# =============================================================================
# REGRESSIONS
# =============================================================================
def ols_hac(y, X, lags=None, const=True):
    """OLS with Newey-West (Bartlett) standard errors. Returns (coef, se, t, resid, r2)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if X.ndim == 1:
        X = X[:, None]
    if const:
        X = np.column_stack([np.ones(len(y)), X])
    n, k = X.shape
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    u = X * e[:, None]
    S = u.T @ u
    for l in range(1, lags + 1):
        w = 1 - l / (lags + 1)
        G = u[l:].T @ u[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(V))
    r2 = 1 - e.var() / y.var()
    return b, se, b / se, e, r2


def grs_test(excess, market_excess):
    """Gibbons-Ross-Shanken (1989) test of alpha = 0 on N assets, one factor.

    Sigma and the factor variance are maximum-likelihood estimators (division by T);
    under i.i.d. Normal residuals, W ~ F(N, T - N - 1) exactly.
    """
    from scipy import stats
    R = np.asarray(excess, float)
    f = np.asarray(market_excess, float)
    T, N = R.shape
    X = np.column_stack([np.ones(T), f])
    B = np.linalg.lstsq(X, R, rcond=None)[0]
    alpha = B[0]
    E = R - X @ B
    Sigma = E.T @ E / T
    mu, s2 = f.mean(), f.var(ddof=0)
    stat = (T - N - 1) / N * (alpha @ np.linalg.solve(Sigma, alpha)) / (1 + mu ** 2 / s2)
    p = 1 - stats.f.cdf(stat, N, T - N - 1)
    return stat, p, alpha, B[1]

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
PALETTE = [MainBlue, IDAred, Forest, Amber, Orange, Purple, Crimson, '#795548', '#17A2B8', '#6F42C1', '#20C997']

# models: FF3 and Carhart use the published three-factor SMB; FF5 uses the five-factor SMB
FF3 = ['Mkt-RF', 'SMB_FF3', 'HML']
CARHART = FF3 + ['MOM']
FF5 = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
FF6 = FF5 + ['MOM']

SEED = 42
MAX_ABS_RET = 0.5        # data-error threshold for the BVB series (daily log return)
END = '2026-07-31'       # last month of the monthly sample used in the chapter (T = 757 months from July 1963)


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def monthly_excess(symbols, start='1999-01-01'):
    """Monthly simple excess returns over RF (one-month T-bill), on the common months."""
    p = prices(symbols).resample('ME').last()
    r = p.pct_change().dropna()
    F = factors('M')
    idx = r.index.intersection(F.index)
    r = r.loc[idx].loc[start:END]
    F = F.loc[r.index]
    return r.sub(F['RF'], axis=0), F


def daily_excess_one(symbol, start=None):
    """Daily excess return of a single asset, on the factor calendar.

    Only days with a price on the previous trading day are kept
    (otherwise the return would span several days and the factors only one).
    """
    s = price(symbol)
    F = factors('D').loc[:END]                         # same sample end as the monthly data
    s = s.loc[s.index.isin(F.index)]
    cal = pd.Series(np.arange(len(F)), index=F.index)
    pos = cal.loc[s.index].values
    r = s.pct_change()
    consecutive = np.r_[False, np.diff(pos) == 1]
    r = r[consecutive].dropna()
    if start:
        r = r.loc[start:]
    Fx = F.loc[r.index]
    return (r - Fx['RF']).rename(symbol), Fx


def daily_excess(symbols, start=None):
    """Compatibility: dictionary {symbol: (excess, factors)} on each asset's calendar."""
    return {s: daily_excess_one(s, start) for s in symbols}

## Analysis

In [ ]:
def sml_data(start='1963-07-31'):
    F = factors('M')
    P = french('p25', 'M').loc[start:END]
    Fx = F.loc[P.index]
    ex = P.sub(Fx['RF'], axis=0)
    mk = Fx['Mkt-RF']
    rows = []
    for c in ex.columns:
        b, se, t, e, r2 = ols_hac(ex[c].values, mk.values)
        rows.append((c, b[1], b[0] * 12, t[0], ex[c].mean() * 12))
    res = pd.DataFrame(rows, columns=['port', 'beta', 'alpha', 't_alpha', 'avg']).set_index('port')
    return res, ex, mk


def pca_sectors():
    S = [s + '.US' for s in SECTORS_ALL]
    r = log_returns(prices(S + ['SPY.US']))
    X = r[S]
    Z = (X - X.mean()) / X.std()
    C = np.corrcoef(Z.values.T)
    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    for k in range(vecs.shape[1]):                      # sign: positive mean loading
        if vecs[:, k].sum() < 0:
            vecs[:, k] *= -1
    pcs = Z.values @ vecs
    corr_pc1_spy = np.corrcoef(pcs[:, 0], r['SPY.US'].values)[0, 1]
    return vals, vecs, r, corr_pc1_spy


def fama_macbeth(ex, F, fac, nw_lags=6):
    """Pass 1: full-sample betas; pass 2: monthly cross-sectional regressions."""
    X = F[fac].values
    B = np.linalg.lstsq(np.column_stack([np.ones(len(X)), X]), ex.values, rcond=None)[0][1:].T   # N x K
    lam = []
    for t in range(len(ex)):
        Z = np.column_stack([np.ones(B.shape[0]), B])
        lam.append(np.linalg.lstsq(Z, ex.values[t], rcond=None)[0])
    lam = np.array(lam)                                   # T x (1+K)
    mean = lam.mean(0)
    se_fm = lam.std(0, ddof=1) / np.sqrt(len(lam))
    se_nw = np.array([ols_hac(lam[:, j], np.zeros((len(lam), 0)), lags=nw_lags)[1][0] for j in range(lam.shape[1])])
    # Shanken (1992) correction: the FM variance already contains Sigma_f / T (factor variation);
    # only the remainder (the idiosyncratic part) is multiplied by (1 + c), c = lambda' Sigma_f^-1 lambda:
    # Var_Sh = (1 + c) (Var_FM - Sigma_f* / T) + Sigma_f* / T, Sigma_f* = Sigma_f bordered with zeros for the intercept
    Sf = np.atleast_2d(np.cov(X.T))
    c = mean[1:] @ np.linalg.solve(Sf, mean[1:])
    sf_diag = np.r_[0.0, np.diag(Sf)] / len(lam)
    se_sh = np.sqrt((1 + c) * (se_fm ** 2 - sf_diag) + sf_diag)
    return pd.DataFrame({'lambda': mean * 12, 'se_fm': se_fm * 12, 'se_nw': se_nw * 12, 'se_shanken': se_sh * 12,
                         'factor_mean': np.r_[np.nan, X.mean(0) * 12]}, index=['const'] + fac)


def grs_sharpe(ex, mk):
    """GRS identity: alpha' Sigma^-1 alpha = SR^2(f, R) - SR^2(f) (ML estimators, division by T)."""
    R = np.asarray(ex, float)
    f = np.asarray(mk, float)
    T, N = R.shape
    Y = np.column_stack([f, R])
    m = Y.mean(0)
    V = np.cov(Y.T, ddof=0)
    sr2_all = m @ np.linalg.solve(V, m)
    sr2_f = f.mean() ** 2 / f.var(ddof=0)
    stat, p, _, _ = grs_test(R, f)
    W_id = (T - N - 1) / N * (sr2_all - sr2_f) / (1 + sr2_f)
    return dict(T=T, N=N, sr_f_m=np.sqrt(sr2_f), sr_all_m=np.sqrt(sr2_all), sr_f_ann=np.sqrt(12 * sr2_f),
                sr_all_ann=np.sqrt(12 * sr2_all), grs=stat, grs_p=p, grs_identity=W_id)


def two_pass_gmm(ex, F, fac, lags=0):
    """Two passes as an exactly identified GMM: moments (R - a - B f) x (1, f) and X'(R - X gamma), X = [1, B].

    Returns gamma (annualised) and the standard errors:
      * 'krs'    : misspecification-robust (Kan, Robotti & Shanken, 2013) -- full derivative,
                   including the term with the pricing errors e = mu - X gamma;
      * 'correct': the same formula with e = 0 (model assumed correct; Jagannathan & Wang, 1998).
    lags > 0: Newey-West (Bartlett) S matrix; lags = 0: serially uncorrelated moments.
    """
    R = np.asarray(ex, float)
    Fm = np.asarray(F[fac], float)
    T, N = R.shape
    K = Fm.shape[1]
    Z = np.column_stack([np.ones(T), Fm])                      # T x (K+1)
    AB = np.linalg.lstsq(Z, R, rcond=None)[0]                  # (K+1) x N: a, B
    B = AB[1:].T                                               # N x K
    X = np.column_stack([np.ones(N), B])
    mu = R.mean(0)
    gam = np.linalg.lstsq(X, mu, rcond=None)[0]
    npar = (K + 1) * N + K + 1

    def unpack(th):
        ab = th[:(K + 1) * N].reshape(K + 1, N)
        return ab, th[(K + 1) * N:]

    def gbar(th, target):
        ab, g = unpack(th)
        E = R - Z @ ab
        g1 = (Z.T @ E / T).ravel()                            # (K+1) x N
        Xb = np.column_stack([np.ones(N), ab[1:].T])
        g2 = Xb.T @ (target - Xb @ g)
        return np.r_[g1, g2]

    th0 = np.r_[AB.ravel(), gam]

    def jac(target):
        J = np.empty((npar, npar))
        h = 1e-6
        for j in range(npar):
            d = np.zeros(npar)
            d[j] = h
            J[:, j] = (gbar(th0 + d, target) - gbar(th0 - d, target)) / (2 * h)
        return J

    # individual moment contributions (their sample mean is zero)
    E = R - Z @ AB
    g1t = (Z[:, :, None] * E[:, None, :]).reshape(T, -1)
    g2t = (R - X @ gam) @ X
    G = np.column_stack([g1t, g2t])
    S = G.T @ G / T
    for l in range(1, lags + 1):
        w = 1 - l / (lags + 1)
        C = G[l:].T @ G[:-l] / T
        S += w * (C + C.T)
    out = {}
    for lab, target in [('krs', mu), ('correct', X @ gam)]:
        Ji = np.linalg.inv(jac(target))
        V = Ji @ S @ Ji.T / T
        out[lab] = np.sqrt(np.diag(V)[-(K + 1):]) * 12
    e = mu - X @ gam
    return dict(gamma=gam * 12, se_krs=out['krs'], se_correct=out['correct'],
                pricing_error_rms=float(np.sqrt((e ** 2).mean()) * 12))


def fm_krs_table(ex, F, fac):
    """Fama-MacBeth (FM, NW 6, Shanken) plus Kan-Robotti-Shanken errors, on the same assets."""
    d = fama_macbeth(ex, F, fac)
    k = two_pass_gmm(ex.values, F, fac)
    d['se_krs'] = k['se_krs']
    d['se_correct'] = k['se_correct']
    return d


def useless_factor_sim(reps=2000, seed=SEED):
    """Kan & Zhang (1999): CAPM + a factor g independent of returns, on the 25 actual portfolios.

    In each replication: g ~ N(0, s^2) i.i.d., two Fama-MacBeth passes; count the rejections
    |t| > 1.96 of lambda_g (FM and Shanken errors) and the rejections of the first-pass Wald test beta_g = 0.
    """
    rng = np.random.default_rng(seed)
    res, ex, mk = sml_data()
    R = ex.values
    T, N = R.shape
    m = mk.values
    rej_fm = rej_sh = rej_w = 0
    tvals = []
    for _ in range(reps):
        gfac = rng.standard_normal(T) * m.std()
        Z = np.column_stack([np.ones(T), m, gfac])
        AB = np.linalg.lstsq(Z, R, rcond=None)[0]
        B = AB[1:].T
        X = np.column_stack([np.ones(N), B])
        lam = np.linalg.lstsq(X, R.T, rcond=None)[0].T        # T x 3
        mean = lam.mean(0)
        se_fm = lam.std(0, ddof=1) / np.sqrt(T)
        Sf = np.cov(np.column_stack([m, gfac]).T)
        c = mean[1:] @ np.linalg.solve(Sf, mean[1:])
        se_sh = np.sqrt((1 + c) * (se_fm[2] ** 2 - Sf[1, 1] / T) + Sf[1, 1] / T)
        t_fm, t_sh = mean[2] / se_fm[2], mean[2] / se_sh
        tvals.append(t_sh)
        rej_fm += abs(t_fm) > 1.96
        rej_sh += abs(t_sh) > 1.96
        # Wald test of beta_g = 0 on all assets (i.i.d. residuals)
        E = R - Z @ AB
        Sig = E.T @ E / T
        vg = np.linalg.inv(Z.T @ Z)[2, 2]
        W = AB[2] @ np.linalg.solve(Sig * vg, AB[2])
        rej_w += W > stats.chi2.ppf(0.95, N)
    tvals = np.array(tvals)
    return dict(reps=reps, rej_fm=rej_fm / reps, rej_shanken=rej_sh / reps, rej_wald_beta=rej_w / reps,
                median_abs_t=float(np.median(np.abs(tvals))))


def lns_r2(start='1963-07-31'):
    """Lewellen, Nagel & Shanken (2010): OLS and GLS cross-sectional R^2, 25 portfolios vs 25 + 30 industries."""
    F = factors('M')
    P25 = french('p25', 'M').loc[start:END]
    I30 = french('ind30', 'M').loc[start:END]
    Fx = F.loc[P25.index]
    sets = {'25': P25, '25+30': pd.concat([P25, I30.add_prefix('IND_')], axis=1)}
    models = {'CAPM': ['Mkt-RF'], 'FF3': FF3, 'FF5': FF5}
    out = {}
    for sname, P in sets.items():
        ex = P.sub(Fx['RF'], axis=0).values
        T, N = ex.shape
        mu = ex.mean(0)
        V = np.cov(ex.T)
        Vi = np.linalg.inv(V)
        for mname, fac in models.items():
            f = Fx[fac].values
            Z = np.column_stack([np.ones(T), f])
            B = np.linalg.lstsq(Z, ex, rcond=None)[0][1:].T
            X = np.column_stack([np.ones(N), B])
            g_ols = np.linalg.lstsq(X, mu, rcond=None)[0]
            e = mu - X @ g_ols
            r2_ols = 1 - e @ e / ((mu - mu.mean()) @ (mu - mu.mean()))
            g_gls = np.linalg.solve(X.T @ Vi @ X, X.T @ Vi @ mu)
            eg = mu - X @ g_gls
            one = np.ones(N)
            mbar = (one @ Vi @ mu) / (one @ Vi @ one)
            d = mu - mbar
            r2_gls = 1 - (eg @ Vi @ eg) / (d @ Vi @ d)
            # LNS restriction for traded factors: lambda = factor mean, no intercept
            ec = mu - B @ f.mean(0)
            r2_c = 1 - ec @ ec / ((mu - mu.mean()) @ (mu - mu.mean()))
            out[f'{mname} | {sname}'] = dict(N=N, r2_ols=r2_ols, r2_gls=r2_gls, r2_constrained=r2_c,
                                             lambda_ols=(g_ols * 12).round(4).tolist())
    return out


def n_factors(X, kmax):
    """Number of factors: Kaiser (eigenvalues > 1), Bai & Ng (2002) IC_p2, Ahn & Horenstein (2013) ER."""
    X = np.asarray(X, float)
    X = (X - X.mean(0)) / X.std(0)
    T, N = X.shape
    ev = np.sort(np.linalg.eigvalsh(X.T @ X / T))[::-1]         # eigenvalues of the correlation matrix
    V = [ev[k:].sum() / N for k in range(kmax + 1)]              # V(k) = (1/NT) sum of squared residuals
    pen = (N + T) / (N * T) * np.log(min(N, T))
    ic = [np.log(V[k]) + k * pen for k in range(kmax + 1)]
    er = [ev[k - 1] / ev[k] for k in range(1, kmax + 1)]
    return dict(T=T, N=N, kaiser=int((ev > 1).sum()), bai_ng=int(np.argmin(ic)),
                ahn_horenstein=int(np.argmax(er) + 1), eig=ev[:kmax + 1].round(3).tolist(),
                er=np.round(er, 2).tolist())


def number_of_factors():
    """Number-of-factors estimators on the sector ETFs (daily) and on 100 portfolios (monthly)."""
    vals, vecs, r, c1 = pca_sectors()
    S = [s + '.US' for s in SECTORS_ALL]
    out = {'sectors': n_factors(r[S].values, 5)}
    P = french('p100', 'M').loc['1963-07-31':END].dropna(axis=1)
    F = factors('M').loc[P.index]
    out['p100'] = n_factors(P.sub(F['RF'], axis=0).values, 8)
    return out


def max_sr2(F, fac):
    m = F[fac].mean().values
    V = np.atleast_2d(np.cov(F[fac].values.T))
    return float(m @ np.linalg.solve(V, m))


def stationary_bootstrap_idx(T, mean_block, rng):
    """Indices for the stationary bootstrap (Politis & Romano, 1994), mean block length mean_block."""
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(T)
    for t in range(1, T):
        idx[t] = rng.integers(T) if rng.random() < 1 / mean_block else (idx[t - 1] + 1) % T
    return idx


def model_comparison(B=2000, mean_block=6, seed=SEED):
    """Barillas & Shanken (2018): maximum squared Sharpe ratio of each model's factors; stationary-bootstrap intervals."""
    rng = np.random.default_rng(seed)
    F = factors('M')
    models = {'CAPM': ['Mkt-RF'], 'FF3': FF3, 'Carhart': CARHART, 'FF5': FF5, 'FF5+MOM': FF6}
    # nested pairs (FF3 - CAPM, FF5+MOM - FF5: in-sample difference >= 0) and non-nested pairs
    # (FF5 - FF3: different SMB in the two models; FF5 - Carhart)
    pairs = [('FF3', 'CAPM'), ('FF5', 'FF3'), ('FF5+MOM', 'FF5'), ('FF5', 'Carhart')]
    out = {}
    for lab, a, b in [('1963-2026', '1963-07-31', END), ('2000-2026', '2000-01-31', END)]:
        Fs = F.loc[a:b]
        T = len(Fs)
        sr2 = {k: max_sr2(Fs, v) for k, v in models.items()}
        boots = {p: [] for p in pairs}
        for _ in range(B):
            Fb = Fs.iloc[stationary_bootstrap_idx(T, mean_block, rng)]
            s = {k: max_sr2(Fb, v) for k, v in models.items()}
            for p in pairs:
                boots[p].append(12 * (s[p[0]] - s[p[1]]))
        # spanning alphas: each factor regressed on the other five (NW)
        full = models['FF5+MOM']
        span = {}
        for c in full:
            bb, se, t, _, _ = ols_hac(Fs[c].values, Fs[[x for x in full if x != c]].values)
            span[c] = dict(alpha=bb[0] * 12, t=t[0])
        out[lab] = dict(T=T, sr_ann={k: np.sqrt(12 * v) for k, v in sr2.items()},
                        sr2_ann={k: 12 * v for k, v in sr2.items()},
                        diff={f'{p[0]} - {p[1]}': dict(est=12 * (sr2[p[0]] - sr2[p[1]]),
                                                        ci=np.percentile(boots[p], [2.5, 97.5]).tolist(),
                                                        share_le0=float(np.mean(np.array(boots[p]) <= 0)))
                              for p in pairs},
                        spanning=span)
    return out


def eiv_attenuation():
    """Errors in variables in pass 2, conditional on the realised factor path:
    plim_N lambda_OLS / lambda_realised = Var(beta) / (Var(beta) + s2_eps / sum_t (f_t - f_bar)^2)."""
    out = {}
    res, ex, mk = sml_data()
    cases = {'25 size x B/M': (ex, mk)}
    S = [s + '.US' for s in SECTORS]
    exs, Fs = monthly_excess(S)
    cases['9 sector ETFs'] = (exs, Fs['Mkt-RF'])
    for k, (e, m) in cases.items():
        R = e.values
        f = np.asarray(m, float)
        T = len(f)
        Z = np.column_stack([np.ones(T), f])
        AB = np.linalg.lstsq(Z, R, rcond=None)[0]
        E = R - Z @ AB
        s2e = (E ** 2).sum(0) / (T - 2)
        noise = s2e.mean() / ((f - f.mean()) ** 2).sum()          # = s2_eps / (T * var_ML(f))
        vb_hat = AB[1].var(ddof=1)
        vb = vb_hat - noise
        out[k] = dict(T=T, N=R.shape[1], var_beta_hat=vb_hat, noise=noise, var_beta=vb,
                      attenuation=vb / (vb + noise), sd_beta=np.sqrt(vb_hat))
    return out


def advanced():
    res, ex, mk = sml_data()
    F = factors('M').loc[ex.index]
    out = dict(grs_sharpe=grs_sharpe(ex.values, mk.values))
    S = [s + '.US' for s in SECTORS]
    exs, Fs = monthly_excess(S)
    out['grs_sharpe_sectors'] = grs_sharpe(exs.values, Fs['Mkt-RF'].values)
    out['fm_krs'] = {k: fm_krs_table(ex, F, v).round(4).reset_index().to_dict(orient='records')
                     for k, v in {'CAPM': ['Mkt-RF'], 'FF3': FF3, 'FF5': FF5}.items()}
    out['useless'] = useless_factor_sim()
    out['lns'] = lns_r2()
    out['nfac'] = number_of_factors()
    out['model_comparison'] = model_comparison()
    out['eiv'] = eiv_attenuation()
    return out


def b9_lns(B=500, seed=SEED):
    """Lewellen-Nagel-Shanken: OLS and GLS R^2, 25 portfolios vs 25 + 30 industries; i.i.d. bootstrap CIs over months."""
    rng = np.random.default_rng(seed)
    F = factors('M')
    P25 = french('p25', 'M').loc['1963-07-31':END]
    I30 = french('ind30', 'M').loc['1963-07-31':END]
    Fx = F.loc[P25.index]
    sets = {'25': P25, '25+30': pd.concat([P25, I30.add_prefix('IND_')], axis=1)}
    models = {'CAPM': ['Mkt-RF'], 'FF3': FF3, 'FF5': FF5}

    def r2s(ex, f):
        T, N = ex.shape
        mu = ex.mean(0)
        Z = np.column_stack([np.ones(T), f])
        Bm = np.linalg.lstsq(Z, ex, rcond=None)[0][1:].T
        X = np.column_stack([np.ones(N), Bm])
        e = mu - X @ np.linalg.lstsq(X, mu, rcond=None)[0]
        r2o = 1 - e @ e / ((mu - mu.mean()) @ (mu - mu.mean()))
        Vi = np.linalg.inv(np.cov(ex.T))
        eg = mu - X @ np.linalg.solve(X.T @ Vi @ X, X.T @ Vi @ mu)
        one = np.ones(N)
        d = mu - (one @ Vi @ mu) / (one @ Vi @ one)
        return r2o, 1 - (eg @ Vi @ eg) / (d @ Vi @ d)

    out = {}
    for sn, P in sets.items():
        ex = P.sub(Fx['RF'], axis=0).values
        T = len(ex)
        for mn, fac in models.items():
            f = Fx[fac].values
            o, gl = r2s(ex, f)
            bo = []
            for _ in range(B):
                i = rng.integers(0, T, T)
                bo.append(r2s(ex[i], f[i]))
            bo = np.array(bo)
            out[f'{mn} | {sn}'] = dict(r2_ols=o, r2_gls=gl, ci_ols=np.percentile(bo[:, 0], [2.5, 97.5]).tolist(),
                                       ci_gls=np.percentile(bo[:, 1], [2.5, 97.5]).tolist())
    return out


def b10_model_comparison():
    """Barillas-Shanken: maximum squared Sharpe ratio of the factors, 1963-2026 and 2000-2026 (stationary bootstrap)."""
    return model_comparison()


def b11_grs_robust(B=2000, lags=6, seed=SEED, draws=False):
    """GRS on 25 portfolios without normality: residual bootstrap (H0 imposed) and GMM Wald with HAC."""
    rng = np.random.default_rng(seed)
    res, ex, mk = sml_data()
    R, f = ex.values, mk.values
    T, N = R.shape
    stat, p, alpha, beta = grs_test(R, f)
    Z = np.column_stack([np.ones(T), f])
    AB = np.linalg.lstsq(Z, R, rcond=None)[0]
    E = R - Z @ AB
    boot = np.empty(B)
    for b in range(B):
        i = rng.integers(0, T, T)
        Rb = np.outer(f, AB[1]) + E[i]                   # alpha = 0 imposed; whole rows (cross-sectional correlation preserved)
        boot[b] = grs_test(Rb, f)[0]
    # GMM Wald: moments e_t x (1, f_t); HAC (Bartlett) variance of (alpha, beta)
    u = (Z[:, :, None] * E[:, None, :]).reshape(T, -1)     # T x 2N, order (1: alpha_1..N, f: beta_1..N)
    S = u.T @ u / T
    for l in range(1, lags + 1):
        C = u[l:].T @ u[:-l] / T
        S += (1 - l / (lags + 1)) * (C + C.T)
    D = np.kron(Z.T @ Z / T, np.eye(N))
    Di = np.linalg.inv(D)
    V = Di @ S @ Di / T
    Va = V[:N, :N]
    W = alpha @ np.linalg.solve(Va, alpha)
    S0 = u.T @ u / T
    V0 = (Di @ S0 @ Di / T)[:N, :N]
    W0 = alpha @ np.linalg.solve(V0, alpha)
    # residuals: skewness and kurtosis (the reason for the bootstrap)
    kurt = float(np.mean(stats.kurtosis(E, axis=0, fisher=False)))
    if draws:
        return boot, stat, E
    return dict(T=T, N=N, grs=stat, p_F=p, p_boot=float((boot >= stat).mean()), n_exceed=int((boot >= stat).sum()),
                p_boot_plus1=float(((boot >= stat).sum() + 1) / (B + 1)), boot_q95=float(np.percentile(boot, 95)),
                wald_hac=W, p_wald_hac=1 - stats.chi2.cdf(W, N), wald_white=W0, p_wald_white=1 - stats.chi2.cdf(W0, N),
                mean_kurtosis=kurt)

## Run

In [ ]:
print(advanced())
print(b9_lns())
print(b11_grs_robust())